In [1]:
! pip install tqdm multiprocess

In [2]:
import numpy as np
import pandas as pd
from multiprocess import Pool

from functions import evaluate_ar, evaluate_multiplier_iid, evaluate_ma
from bstrapping.synthetic_time_series.moving_average import MovingAverage

In [3]:
mean = 4  # mean of the time series
alpha = 0.05

# Names of the stochastic processes

# Dependence coefficients of the stochastic processes, i.e. of the moving average processes
dependence_coefficients = [
    np.array([0]),
    np.array([0.5]),
    np.array([0.5 ** i for i in range(1, 3)]),
    np.array([0.5 ** i for i in range(1, 10)]),
    np.array([0.5 ** i for i in range(1, 15)]),
    np.array([0.5 ** i for i in range(1, 20)]),
]

names_dependence_coefficients = [
    "iid",
    "MA(1)",
    "MA(2)",
    "MA(10)",
    "MA(15)",
    "MA(20)",
]

list_name_weights = ['AR',
                     ]

## Scripts

In [90]:
benchmark = []
means_of_variance = []
std_of_variance = []
coverage_probability = []

In [91]:
# 1000, 2000, 5000, 10000, 150000
sample_size = 1000
runs = 250
index_dependence = 1

time_series = MovingAverage(mean=mean, parameters=dependence_coefficients[index_dependence])

print("Time series: ",names_dependence_coefficients[index_dependence], "\nSample size: ", sample_size)

Time series:  MA(1) 
Sample size:  1000


In [92]:
%%time
samples = [
    MovingAverage(mean=mean, parameters=dependence_coefficients[index_dependence]).generate_samples(sample_size)
    for _ in range(runs)]

CPU times: user 2.66 s, sys: 15.6 ms, total: 2.67 s
Wall time: 2.67 s


## Benchmark bootstraps

In [93]:
%%time
%%capture
# benchmark AR bootstrap
p = Pool()

evaluations_ar = p.map(lambda sample: evaluate_ar(sample, alpha=alpha,number_bootstrap_samples=250), samples)

result = np.array(evaluations_ar)

coverage_probability.append(np.sum(result.T[1]) / runs)
means_of_variance.append(np.mean(result.T[0]))
std_of_variance.append(np.std(result.T[0]))


CPU times: user 209 ms, sys: 195 ms, total: 404 ms
Wall time: 17.9 s


## Concatination result

In [94]:
benchmark = [names_dependence_coefficients[index_dependence], time_series.asymptotic_variance,
             sample_size] + means_of_variance + std_of_variance + [1 - alpha] + coverage_probability

In [95]:
benchmark = pd.DataFrame([benchmark], columns=pd.MultiIndex.from_tuples([("Stochastic process", ""),
                                                             ("mean", "Asymptotic variance"),
                                                             ("Sample size", "")] +
                                                            [("mean", name,) for name in list_name_weights] +
                                                            [("std", name,) for name in list_name_weights] +
                                                            [("In confidence interval", "Confidence level")]
                                                            +
                                                            [("In confidence interval", name,) for name in
                                                             list_name_weights]
                                                            )).set_index(["Sample size"])

In [96]:
benchmark

Stochastic process                mean                std  \
                               Asymptotic variance        AR       AR   
Sample size                                                             
1000                     MA(1)                2.25  2.229676  0.41301   

            In confidence interval        
                  Confidence level    AR  
Sample size                               
1000                          0.95  0.94

In [97]:
#benchmark.to_csv(f"./data/benchmark_{sample_size}_{names_dependence_coefficients[index_dependence]}_ar.csv")
#benchmark.to_pickle(f"./data/benchmark_{sample_size}_{names_dependence_coefficients[index_dependence]}_ar.pkl")

In [98]:
#whole_benchmark = benchmark

In [99]:
whole_benchmark = pd.concat((whole_benchmark,benchmark))

In [100]:
whole_benchmark

Stochastic process                mean                 std  \
                               Asymptotic variance        AR        AR   
Sample size                                                              
10000                    MA(1)                2.25  2.232946  0.291841   
5000                     MA(1)                2.25  2.221108  0.284993   
2000                     MA(1)                2.25  2.268564  0.387044   
1000                     MA(1)                2.25  2.229676  0.413010   

            In confidence interval         
                  Confidence level     AR  
Sample size                                
10000                         0.95  0.920  
5000                          0.95  0.952  
2000                          0.95  0.960  
1000                          0.95  0.940